In [1]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install roboflow


  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-uq1eu2zd
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-uq1eu2zd
  Resolved https://github.com/facebookresearch/detectron2.git to commit 1e3e13bbf607b54f62205c4c33922521822fb298
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.2/105.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.9 MB/s eta 0:00:00
  Created wheel for 

In [2]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API = userdata.get('ROBOFLOW_API')
rf = Roboflow(api_key=ROBOFLOW_API)
project = rf.workspace("udy").project("visdrone-qr0vn")
version = project.version(2)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Visdrone-2 in coco:: 100%|██████████| 8633/8633 [00:12<00:00, 698.20it/s]


In [3]:
dataset.location

'/content/Visdrone-2'

In [4]:
import os
from detectron2.data.datasets import register_coco_instances
from detectron2.config import get_cfg
from detectron2 import model_zoo

# 1. Khai báo 2 thư mục train và valid
data_dir = dataset.location
register_coco_instances("visdrone_train", {}, f"{data_dir}/train/_annotations.coco.json", f"{data_dir}/train")
register_coco_instances("visdrone_val", {}, f"{data_dir}/valid/_annotations.coco.json", f"{data_dir}/valid")
# 2. Tạo file Config cho Cascade R-CNN
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.MASK_ON = False  # Chỉ lấy Bounding Box
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml")
cfg.DATASETS.TRAIN = ("visdrone_train",)
cfg.DATASETS.TEST = ("visdrone_val",)
cfg.DATALOADER.NUM_WORKERS = 2
# Thông số học (Training)
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 2e-3
cfg.SOLVER.MAX_ITER = 50000
cfg.SOLVER.STEPS = (40000, 47500)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 10  # VisDrone có 10 nhóm

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print('config done')

config done


pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [5]:
import detectron2.data.transforms as T
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader, DatasetMapper

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        custom_augmentations = [
            T.ResizeShortestEdge(short_edge_length=(640, 672, 704, 736, 768, 800), max_size=1333, sample_style="choice"),
            T.RandomCrop("relative_range", (0.5, 0.5)),
            T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            T.RandomBrightness(0.8, 1.2),
            T.RandomContrast(0.8, 1.2),
        ]
        mapper = DatasetMapper(cfg, is_train=True, augmentations=custom_augmentations)
        return build_detection_train_loader(cfg, mapper=mapper)


In [6]:
trainer = CustomTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[10/06 10:13:52 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_480dd8.pkl: 288MB [00:01, 178MB/s]                           
roi_heads.box_predictor.0.cls_score.{bias, weight}
roi_heads.box_predictor.1.cls_score.{bias, weight}
roi_heads.box_predictor.2.cls_score.{bias, weight}
  roi_heads.mask_head.mask_fcn1.{bias, weight}
  roi_heads.mask_head.mask_fcn2.{bias, weight}
  roi_heads.mask_head.mask_fcn3.{bias, weight}
  roi_heads.mask_head.mask_fcn4.{bias, weight}
  roi_heads.mask_head.deconv.{bias, weight}
  roi_heads.mask_head.predictor.{bias, weight}


[10/06 10:13:56 d2.engine.train_loop]: Starting training from iteration 0


torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
W1006 10:14:02.253000 2114 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[10/06 10:14:22 d2.utils.events]:  eta: 13:49:48  iter: 19  total_loss: 9.626  loss_cls_stage0: 2.391  loss_box_reg_stage0: 0.4807  loss_cls_stage1: 2.289  loss_box_reg_stage1: 0.7385  loss_cls_stage2: 2.322  loss_box_reg_stage2: 0.6492  loss_rpn_cls: 0.4203  loss_rpn_loc: 0.3618    time: 0.9962  last_time: 1.1317  data_time: 0.0558  last_data_time: 0.0167   lr: 3.9962e-05  max_mem: 4029M
[10/06 10:14:50 d2.utils.events]:  eta: 14:37:18  iter: 39  total_loss: 8.048  loss_cls_stage0: 1.939  loss_box_reg_stage0: 0.4781  loss_cls_stage1: 1.845  loss_box_reg_stage1: 0.7129  loss_cls_stage2: 1.798  loss_box_reg_stage2: 0.6081  loss_rpn_cls: 0.2995  loss_rpn_loc: 0.3639    time: 1.0374  last_time: 1.1397  data_time: 0.0177  last_data_time: 0.0151   lr: 7.9922e-05  max_mem: 4734M
[10/06 10:15:13 d2.utils.events]:  eta: 14:41:33  iter: 59  total_loss: 5.605  loss_cls_stage0: 1.079  loss_box_reg_stage0: 0.5099  loss_cls_stage1: 1.011  loss_box_reg_stage1: 0.743  loss_cls_stage2: 0.8953  loss_bo

KeyboardInterrupt: 

In [ ]:
!pip install huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)

repo_id = "huynoob1312/detectron2-cascade-rcnn-visdrone"
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

model_path = "./output/model_final.pth"
if os.path.exists(model_path):
    api.upload_file(
        path_or_fileobj=model_path,
        path_in_repo="model_final.pth",
        repo_id=repo_id,
    )
else:
    print(f"❌ Lỗi: Không tìm thấy file {model_path}!")

for file_name in ["config.yaml", "metrics.json"]:
    file_path = f"./output/{file_name}"
    if os.path.exists(file_path):
        print(f"Đang đẩy file {file_name}...")
        api.upload_file(
            path_or_fileobj=file_path,
            path_in_repo=file_name,
            repo_id=repo_id,
        )